In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, window, count
from fraud_streaming.spark.schema import transaction_schema
from pyspark.sql import Window, functions as F                                                                                   
from config import processed_data_path, checkpoint_path
from data_processing import process_and_write_batch
import sys                                                                                                   
sys.path.append("..") 
from fraud_streaming.redis.redis_client import redis_client

In [2]:
spark = SparkSession.builder \
        .appName("KafkaIncrementalBatch") \
        .master("local[*]") \
        .config(
            "spark.jars.packages", 
            "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0"
        ) \
        .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Spark version: 4.2.0


In [ ]:
df_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "transaction-stream")
    .option("startingOffsets", "earliest")
    .option("maxOffsetsPerTrigger", 10000)
    .load()
)

parsed_stream_df = (
    df_stream
    .select(
        from_json(
            col("value").cast("string"), 
            transaction_schema
        ).alias("data")
    )
    .select("data.*")
)

query = (
    parsed_stream_df.writeStream
    .foreachBatch(process_and_write_batch)
    .option("checkpointLocation", checkpoint_path.as_posix())
    # .trigger(processingTime="10 seconds")
    .start()
)

query.awaitTermination()